# Confusion Matrix & Cross Validation

- Confusion_Matrix: table of TP, TN, FP, FN showing correct/incorrect predictions
- Cross_Validation: tests model on multiple train/test splits (folds) for a more reliable accuracy estimate
- Dataset: 50_Startups.csv
- Note: Confusion Matrix needs categories, but Profit is a number.
  So we convert Profit into a category: "High Profit" (1) if above the median, else "Low Profit" (0)


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("/content/50_Startups[1].csv")
df_encoded = pd.get_dummies(df, columns=["State"], drop_first=True)

# Convert Profit into a binary class: 1 = High Profit (above median), 0 = Low Profit
median_profit = df_encoded["Profit"].median()
df_encoded["High_Profit"] = (df_encoded["Profit"] > median_profit).astype(int)

X = df_encoded.drop(["Profit", "High_Profit"], axis=1)
Y = df_encoded["High_Profit"]
df_encoded.head()

,R&D Spend,Administration,Marketing Spend,Profit,State_Florida,State_New York,High_Profit
0,165349.20,136897.80,471784.10,192261.83,False,True,1
1,162597.70,151377.59,443898.53,191792.06,False,False,1
2,153441.51,101145.55,407934.54,191050.39,True,False,1
3,144372.41,118671.85,383199.62,182901.99,False,True,1
4,142107.34,91391.77,366168.42,166187.94,True,False,1


In [3]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

print("Train size:", X_train.shape[0])
print("Test size :", X_test.shape[0])

Train size: 40
Test size : 10


In [4]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [5]:
model = LogisticRegression()
model.fit(X_train, Y_train)

LogisticRegression()

In [6]:
Y_pred = model.predict(X_test)

print(pd.DataFrame({"Actual": Y_test, "Predicted": Y_pred}))

    Actual  Predicted
13       1          1
39       0          0
30       0          0
45       0          0
17       1          1
48       0          0
26       0          0
25       0          0
32       0          0
19       1          0


In [7]:
cm = confusion_matrix(Y_test, Y_pred)
acc = accuracy_score(Y_test, Y_pred)

print("Confusion Matrix:\n", cm)
print("Accuracy:", round(acc, 4))

Confusion Matrix:
 [[7 0]
 [1 2]]
Accuracy: 0.9


In [8]:
cv_scores = cross_val_score(model, X_train, Y_train, cv=5)

print("CV Scores:", cv_scores)
print("Mean Accuracy:", round(cv_scores.mean(), 4))
print("Std Deviation :", round(cv_scores.std(), 4))

CV Scores: [1.    0.875 0.875 1.    1.   ]
Mean Accuracy: 0.95
Std Deviation : 0.0612


## Conclusion
- Profit was converted into High/Low classes (split at the median) so a Confusion Matrix could be used
- Confusion Matrix: diagonal values = correct predictions, off-diagonal = errors (FP/FN)
- Accuracy: overall % of correct predictions on the test set
- Cross Validation: trains/tests the model on 5 different folds of the data (5 used here since the dataset only has 50 rows)
- Mean CV accuracy gives a more reliable performance estimate than a single train-test split
- Low std deviation across folds means the model is stable/consistent

Save as Confusion_Matrix_Cross_Validation.ipynb
Upload to 03_Machine_Learning/04_Model_Evaluation/
